# 04_從零寫一個CNN_MNIST.py
**電腦視覺與深度學習 › Day18 CNN + Teachable Machine + MediaPipe**

這支程式自己蓋一個小小的卷積神經網路,學會辨認 0 到 9 的手寫數字。它把 Teachable Machine 幫你做的事情攤開來看。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/1)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY9"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '04_從零寫一個CNN_MNIST.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 目標與匯入
目標是用最少的程式碼達到 98% 以上的準確率。匯入 TensorFlow、Keras 的 layers 和 models,也設定中文字型。

In [ ]:
# 04 — 從零寫一個 CNN，訓練 MNIST 手寫數字辨識
#
# 目的：打開 CNN 這個黑箱，看它到底長怎樣、怎麼訓練
# 用最少的程式碼達到 >98% 準確率
#
# 需要：pip install tensorflow matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei"]

### 載入並整理 MNIST
MNIST 有 6 萬張訓練圖、1 萬張測試圖,每張 28 乘 28 的灰階。除以 255 把數值縮到 0 到 1,再加上一個通道維度,變成 28 乘 28 乘 1。

In [ ]:
plt.rcParams["axes.unicode_minus"] = False

# ====== 1) 載入資料 ======
# MNIST 是 keras 內建：60000 張訓練 + 10000 張測試，每張 28x28 灰階
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f"訓練資料：{x_train.shape}，測試資料：{x_test.shape}")

# 正規化 [0, 255] → [0, 1]，加上 channel 維度 (28,28) → (28,28,1)
x_train = (x_train.astype("float32") / 255.0)[..., None]

### 疊出 CNN
結構是:卷積、池化、卷積、池化、攤平、全連接、最後 softmax 輸出 10 個機率。尺寸變化是 28 到 26 再到 13,然後 11 再到 5。第一層 16 個濾波器找線條,第二層 32 個找轉角和彎曲。

In [ ]:
x_test  = (x_test.astype("float32")  / 255.0)[..., None]

# ====== 2) 蓋一個小 CNN ======
# 結構：Conv → Pool → Conv → Pool → Flatten → Dense → Softmax
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # 第 1 層卷積：16 個 3x3 filter，找低階特徵（線條、邊）
    layers.Conv2D(16, kernel_size=3, activation="relu"),
    layers.MaxPool2D(pool_size=2),                       # 28→13

    # 第 2 層卷積：32 個 3x3 filter，找中階特徵（角、彎曲）
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPool2D(pool_size=2),                       # 13→5

    # 攤平 → 全連接 → 10 類機率
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

### 看模型摘要
summary 會印出每一層的輸出形狀和參數量,總共大約 5 萬 6 千多個參數,大部分在全連接層。

### 編譯與訓練
compile 設定學習方法 adam、損失函數和要看的指標。fit 開始訓練:跑 5 輪,每次看 128 張,拿 10% 當驗證集。終端機的數字一直變是正常的,不是當機。

In [ ]:
model.summary()   # 看每層 output shape 跟參數量

# ====== 3) 編譯 + 訓練 ======
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",     # 標籤是整數 0~9
    metrics=["accuracy"],
)

### 用測試集考試
訓練完,用模型從沒看過的 1 萬張測試圖評估,印出測試集準確率。這才是真正的成績。

In [ ]:
# 5 epochs 通常就 >98%；CPU 上約 1~3 分鐘
history = model.fit(
    x_train, y_train,
    epochs=5, batch_size=128,
    validation_split=0.1,
    verbose=1,
)

# ====== 4) 測試集評估 ======
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

### 畫學習曲線
左圖是準確率、右圖是損失,各有訓練和驗證兩條線。如果訓練一直變好、驗證卻變差,就是過擬合的訊號。

In [ ]:
print(f"\n測試集準確率：{test_acc * 100:.2f}%")

# ====== 5) 畫學習曲線 ======
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["accuracy"],     label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("準確率"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history.history["loss"],     label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("損失 (loss)"); axes[1].set_xlabel("epoch"); axes[1].legend()

### 看看預測結果
拿 10 張測試圖給模型猜,猜對的標題是綠色、猜錯是紅色。

In [ ]:
plt.tight_layout(); plt.show()

# ====== 6) 秀 10 張測試圖 + 預測 ======
sample = x_test[:10]
preds  = model.predict(sample, verbose=0).argmax(axis=1)

plt.figure(figsize=(15, 2.5))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(sample[i].squeeze(), cmap="gray")
    color = "green" if preds[i] == y_test[i] else "red"
    plt.title(f"預測:{preds[i]}\n真實:{y_test[i]}", color=color, fontsize=10)
    plt.axis("off")

### 存模型(可選)
把註解打開就能把模型存成檔案,下次直接載入,不用重新訓練。手寫辨識那支 GUI 程式就是這樣做的。

In [ ]:
plt.tight_layout(); plt.show()

# ====== 7) 存下訓練好的模型（可選）======
# model.save("mnist_cnn.h5")
# print("模型已存到 mnist_cnn.h5")

## 小結
你親手訓練了一個 CNN,也看懂了 epoch、準確率和損失。下一支遷移學習會直接借用 Google 訓練好的大模型。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/1) 看逐句拆解、做練習。